# 13 · Трекинг объектов / Object tracking
**RU.** MOT15: реальные уличные видеопоследовательности TUD-Campus (71 кадр, 25 FPS) и TUD-Stadtmitte (179 кадров, 25 FPS). Наблюдение — кадр, вход — RGB/gray и публичные детекции x/y/w/h/confidence, цель — рамка и постоянный ID человека. [Официальный архив](https://motchallenge.net/data/MOT15/). Используйте для учебного исследования с указанием MOTChallenge и авторов TUD, соблюдая условия исходных наборов. Данные не перепубликуются в GitHub.
**EN.** MOT15 contains real street videos: TUD-Campus (71 frames, 25 FPS) and TUD-Stadtmitte (179 frames, 25 FPS). Observations are video frames, inputs are pixels and public x/y/w/h/confidence detections, targets are boxes and persistent pedestrian IDs. Use the official archive for educational research with MOTChallenge/TUD attribution and respect original-source terms. Dataset files are not redistributed here.

**RU.** Первый запуск скачивает 1.31 ГБ, затем использует кэш; CPU, NumPy, SciPy, OpenCV, matplotlib. Никаких случайных кругов вместо людей. TUD-Campus используем для настройки, TUD-Stadtmitte — один раз после фиксации параметров. Обе последовательности принадлежат официальному train: наш test — учебный, не скрытый benchmark. Соседние кадры не перемешиваем. GT разрешён для оценки, но не подаётся трекеру. Наш IoU-трекер не является реализацией Deep SORT.
**EN.** First run downloads 1.31 GB, then reuses the cache. CPU dependencies: NumPy, SciPy, OpenCV, matplotlib. Tune on TUD-Campus and evaluate TUD-Stadtmitte once after freezing parameters. Both are official training sequences; our held-out sequence is a teaching split, not the hidden benchmark. Never shuffle adjacent frames. Use ground truth only for evaluation, not as tracker input. The IoU tracker is not a Deep SORT implementation.

## ДЗ / Homework
RU: Четыре функции по 25 баллов. Заполните имя/группу, выполните весь ноутбук и сдайте .ipynb с выводами. Срок — в матрице по группе, штрафы применяет внешний проверяющий. EN: Four functions worth 25 points each. Fill identity/group, run all cells, submit the notebook with outputs. The group matrix defines deadlines; the external grader applies penalties.

In [ ]:
student_first=""  # RU: укажите имя; EN: enter first name.
student_last=""  # RU: укажите фамилию; EN: enter last name.
student_group=""  # RU: укажите группу; EN: enter group.
assignment_id="TRK-13"  # RU: сохраняйте ID задания; EN: preserve the assignment ID.

In [ ]:
from pathlib import Path  # RU: задаём пути к кэшу; EN: define cache paths.
import urllib.request, zipfile, hashlib, json  # RU: загружаем и проверяем архив; EN: download and verify the archive.
import numpy as np  # RU: обрабатываем координаты; EN: process coordinates.
import cv2  # RU: вычисляем оптический поток; EN: compute optical flow.
import matplotlib.pyplot as plt  # RU: показываем реальные кадры; EN: display real frames.
from scipy.optimize import linear_sum_assignment  # RU: решаем задачу назначения; EN: solve linear assignment.
data=next((p for p in [Path('build/data'),Path('../build/data'),Path('data')] if (p/'MOT15.zip').exists()),Path('data'))  # RU: находим существующий архив; EN: locate an existing archive.
data.mkdir(parents=True,exist_ok=True)  # RU: создаём кэш при необходимости; EN: create cache if needed.
archive=data/'MOT15.zip'  # RU: задаём имя официального архива; EN: name the official archive.
if not archive.exists():  # RU: скачиваем только отсутствующий файл; EN: download only a missing file.
    urllib.request.urlretrieve('https://motchallenge.net/data/MOT15.zip',archive)  # RU: используем официальный сервер; EN: use the official server.
h=hashlib.sha256()  # RU: создаём потоковую контрольную сумму; EN: initialize streaming checksum.
with archive.open('rb') as f:  # RU: не загружаем весь архив в память; EN: avoid loading the entire archive into memory.
    for chunk in iter(lambda:f.read(1048576),b''): h.update(chunk)  # RU: хешируем блоки по мегабайту; EN: hash one-megabyte chunks.
assert h.hexdigest()=='4197530e8bff30fc9fb55565280b8235ab3cc9d16a2eca3ba95e7d5dcae3a5b4'  # RU: проверяем зафиксированные данные; EN: verify pinned data.
with zipfile.ZipFile(archive) as z:  # RU: открываем архив без произвольной распаковки; EN: open without unrestricted extraction.
    for seq in ['TUD-Campus','TUD-Stadtmitte']:  # RU: выбираем две учебные сцены; EN: select two teaching scenes.
        prefix='MOT15/train/'+seq+'/'  # RU: ограничиваем допустимый путь; EN: restrict the allowed path.
        for name in z.namelist():  # RU: перебираем имена внутри архива; EN: inspect archive members.
            if name.startswith(prefix) and not name.endswith('/') and '..' not in Path(name).parts:  # RU: допускаем только безопасные файлы сцены; EN: allow safe scene files only.
                dest=data/name  # RU: строим локальный путь; EN: construct the local path.
                dest.parent.mkdir(parents=True,exist_ok=True)  # RU: создаём подкаталог; EN: create the subdirectory.
                if not dest.exists(): dest.write_bytes(z.read(name))  # RU: извлекаем файл один раз; EN: extract each file once.
def load_sequence(seq):  # RU: читаем наблюдения и метки отдельно; EN: load observations and labels separately.
    root=data/'MOT15/train'/seq  # RU: выбираем каталог сцены; EN: select the scene directory.
    det=np.loadtxt(root/'det/det.txt',delimiter=',')  # RU: читаем публичные детекции; EN: read public detections.
    gt=np.loadtxt(root/'gt/gt.txt',delimiter=',')  # RU: читаем GT только для оценки; EN: read GT only for evaluation.
    det[:,2:4]-=1;gt[:,2:4]-=1  # RU: переводим x/y из 1-based в 0-based; EN: convert x/y from one-based to zero-based.
    frames=sorted((root/'img1').glob('*.jpg'))  # RU: сохраняем временной порядок; EN: preserve temporal order.
    return det,gt,frames  # RU: возвращаем данные с разными ролями; EN: return data with distinct roles.
det,gt,frames=load_sequence('TUD-Campus')  # RU: начинаем со сцены разработки; EN: start with the development scene.
a=cv2.imread(str(frames[0]),0);b=cv2.imread(str(frames[1]),0)  # RU: читаем соседние реальные кадры; EN: read adjacent real frames.
p0=cv2.goodFeaturesToTrack(a,150,.01,7)  # RU: находим исходные углы; EN: detect initial corners.
p1,s1,_=cv2.calcOpticalFlowPyrLK(a,b,p0,None,winSize=(21,21),maxLevel=3)  # RU: переносим точки вперёд; EN: track points forward.
pback,s2,_=cv2.calcOpticalFlowPyrLK(b,a,p1,None,winSize=(21,21),maxLevel=3)  # RU: возвращаем точки назад; EN: track points backward.


## 1 · Границы кадра / Image boundaries
RU: `clip_boxes(boxes,width,height)` получает (N,4) x/y/w/h и возвращает рамки в том же формате после обрезки по [0,width]×[0,height]. Координаты вещественные, без +1 к площади. Отрицательные размеры входа и неположительные width/height вызывают ValueError. Пустой вход даёт (0,4), исходник не менять. EN: Clip float x/y/w/h boxes to image bounds without inclusive-pixel +1. Reject negative box sizes or nonpositive image dimensions. Preserve empty shape and input data.

In [ ]:
def clip_boxes(boxes,width,height):  # RU: ограничиваем рамки изображением; EN: clip boxes to image bounds.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 2 · Модель движения / Motion model
RU: `predict_centres(previous,current,dt_observed,dt_future)` для двух (N,2) массивов возвращает current+(current−previous)*dt_future/dt_observed. Размерности должны совпадать, dt_observed>0, dt_future>=0, иначе ValueError. Времена в секундах; размеры кадра не являются скоростью. EN: Predict centres under constant velocity, validating equal (N,2) shapes, positive observed interval and nonnegative forecast interval. Times are in seconds. Explain the limitation at pedestrian turns.

In [ ]:
def predict_centres(previous,current,dt_observed,dt_future):  # RU: экстраполируем постоянную скорость; EN: extrapolate constant velocity.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 3 · Назначение с порогом / Gated assignment
RU: `gated_assignment(cost,limit)` вернёт список (row,col) по возрастанию row. Разрешены только конечные стоимости 0<=cost<=limit, каждый индекс используется максимум один раз. Сначала максимизировать число допустимых пар, затем минимизировать суммарную стоимость. Используйте Hungarian с достаточно большим штрафом для запрещённых пар, после назначения их удалите. Пустая матрица возвращает []. limit должен быть конечным и >=0, иначе ValueError. EN: Return one-to-one assignments sorted by row. Allow finite nonnegative costs within the limit. Maximize valid pair count before minimizing total cost. Use Hungarian with a sufficiently large forbidden-pair cost, then remove prohibited pairs. Validate the limit and support empty matrices.

In [ ]:
def gated_assignment(cost,limit):  # RU: решаем ограниченное назначение; EN: solve gated assignment.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 4 · Непрерывные фрагменты / Continuous fragments
RU: `track_runs(rows)` принимает (frame,id), удаляет дубликаты, для каждого ID сортирует кадры и считает отрезки последовательных номеров. Вернуть dict tracks,runs,longest. Пустой вход: нули. Смена номера ID создаёт другую траекторию. Оцените реальную GT и объясните, почему длинный run сам по себе не доказывает верный ID. EN: Deduplicate (frame,id) observations and count contiguous frame runs separately per ID. Return track count, run count and longest run length. Empty input returns zeros. Explain why continuity alone does not prove identity correctness.

In [ ]:
def track_runs(rows):  # RU: измеряем непрерывные участки наблюдений; EN: measure contiguous observation runs.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

In [ ]:
score=0;checks=[]  # RU: начинаем независимое начисление баллов; EN: initialize independent scoring.
try:  # RU: проверяем задание 1 независимо; EN: test task 1 independently.
    boxes=det[det[:,0]==1,2:6].copy()  # RU: используем реальные детекции; EN: use real detections.
    original=boxes.copy();out=clip_boxes(boxes,640,480)  # RU: сохраняем контроль исходника; EN: preserve the input for comparison.
    assert np.array_equal(boxes,original) and out.shape==boxes.shape  # RU: проверяем отсутствие побочных изменений; EN: check absence of side effects.
    for old,new in zip(boxes,out):  # RU: независимо проверяем каждую рамку; EN: independently verify each box.
        x,y,w,h=old;lo=np.maximum([x,y],0);hi=np.minimum([x+w,y+h],[640,480])  # RU: вычисляем пересечение с кадром; EN: compute intersection with the frame.
        assert np.allclose(new,np.r_[np.clip(lo,0,[640,480]),np.maximum(hi-lo,0)])  # RU: сверяем результат; EN: verify the result.
    assert clip_boxes([],640,480).shape==(0,4)  # RU: проверяем пустой вход; EN: check empty input.
    score+=25;checks.append({'task':1,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':1,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 2 независимо; EN: test task 2 independently.
    ids=np.intersect1d(gt[gt[:,0]==1,1],gt[gt[:,0]==2,1])  # RU: сопоставляем реальные GT по ID; EN: align real GT by ID.
    a=np.array([gt[(gt[:,0]==1)&(gt[:,1]==i)][0,2:6] for i in ids]);b=np.array([gt[(gt[:,0]==2)&(gt[:,1]==i)][0,2:6] for i in ids])  # RU: извлекаем соседние наблюдения; EN: extract adjacent observations.
    a=a[:,:2]+a[:,2:]/2;b=b[:,:2]+b[:,2:]/2  # RU: переводим рамки в центры; EN: convert boxes to centres.
    assert np.allclose(predict_centres(a,b,.04,.08),b+2*(b-a))  # RU: проверяем разные интервалы времени; EN: check unequal time intervals.
    assert np.allclose(predict_centres(a,b,.04,0),b)  # RU: нулевой горизонт сохраняет текущий центр; EN: zero horizon preserves current centres.
    try: predict_centres(a,b,0,.04)  # RU: проверяем защиту от нулевого dt; EN: test zero-dt validation.
    except ValueError: pass  # RU: принимаем ожидаемый отказ; EN: accept the expected rejection.
    else: raise AssertionError('Zero observed interval must fail')  # RU: отклоняем неверный контракт; EN: reject an invalid contract.
    score+=25;checks.append({'task':2,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':2,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 3 независимо; EN: test task 3 independently.
    boxes=det[det[:,0]==1,2:6];centres=boxes[:,:2]+boxes[:,2:]/2  # RU: используем реальные центры; EN: use real centres.
    cost=np.linalg.norm(centres[:,None]-centres[None,:],axis=2)  # RU: строим расстояния в пикселях; EN: construct pixel distances.
    assert gated_assignment(cost,0)==[(i,i) for i in range(len(cost))]  # RU: точные совпадения должны сохраниться; EN: exact matches must be preserved.
    assert gated_assignment(cost[:0],100)==[]  # RU: проверяем пустую матрицу; EN: check empty matrix.
    changed=cost.copy();changed[0,:]=np.nan  # RU: моделируем недоступные измерения реальной строки; EN: mark one real measurement row unavailable.
    ans=gated_assignment(changed,0)  # RU: запрещаем недоступные пары; EN: prohibit unavailable pairs.
    assert len(ans)==len(cost)-1 and all(i!=0 for i,j in ans)  # RU: пропуск не должен ломать остальные совпадения; EN: missing data must preserve other matches.
    score+=25;checks.append({'task':3,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':3,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 4 независимо; EN: test task 4 independently.
    rows=gt[:,:2].astype(int)  # RU: используем настоящую временную разметку; EN: use real temporal annotations.
    result=track_runs(rows)  # RU: измеряем GT-траектории; EN: measure GT trajectories.
    assert result['tracks']==len(np.unique(rows[:,1]))  # RU: сверяем число идентификаторов; EN: verify identity count.
    assert track_runs(np.vstack([rows,rows]))==result  # RU: дубликаты не должны удлинять трек; EN: duplicates must not lengthen a track.
    assert track_runs(rows[::-1])==result and track_runs([])==dict(tracks=0,runs=0,longest=0)  # RU: проверяем порядок и пустой ввод; EN: check ordering and empty input.
    expected=[]  # RU: независимо собираем длины фрагментов; EN: independently collect run lengths.
    for identity in np.unique(rows[:,1]):  # RU: проверяем каждый реальный ID; EN: inspect each real ID.
        times=np.unique(rows[rows[:,1]==identity,0]);parts=np.split(times,np.flatnonzero(np.diff(times)>1)+1)  # RU: делим по временным разрывам; EN: split at temporal gaps.
        expected.extend(map(len,parts))  # RU: сохраняем длины отрезков; EN: store segment lengths.
    assert result['runs']==len(expected) and result['longest']==max(expected)  # RU: сверяем итоговые метрики; EN: verify aggregate metrics.
    score+=25;checks.append({'task':4,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':4,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
print(checks)  # RU: показываем результат каждого блока; EN: show each block result.
print(json.dumps({'name':(student_first+' '+student_last).strip(),'group':student_group,'assignment':assignment_id,'score':score,'max_score':100},ensure_ascii=False))  # RU: печатаем финальный JSON для проверяющего; EN: print final JSON for the checker.